# Coding Agent — Step-by-Step Debugger

이 노트북은 LangGraph 기반 코딩 에이전트의 각 노드를 **개별적으로 실행**하고,
단계 사이의 상태(state)를 검사하며, LLM 프롬프트/응답을 확인할 수 있는 디버깅 도구입니다.

## 사용 방법
1. 셀을 **위에서 아래로 순서대로** 실행하세요
2. 각 노드 실행 후 출력되는 **state diff**를 확인하세요
3. 필요하면 `GOAL`이나 중간 state를 직접 수정하세요
4. Step 13의 반복 루프 셀을 재실행하여 test→fix 사이클을 반복할 수 있습니다

## 사전 요구사항
- `pip install -r requirements.txt` 완료
- `.env` 파일에 `OPENAI_API_KEY`, `MODEL_NAME` 설정
- (선택) RAG 사용 시 `CODEBASE_DIR` 환경변수 설정

## 파이프라인 구조 (10개 노드)
```
goal_analyzer → rag_retriever → planner → [backend_coder | frontend_coder]
                                                    ↓
                                              test_writer → tester
                                                    ↑           ↓
                                                 fixer ← error_analyzer
                                                    ↓
                                              summarizer → END
```

> **참고**: Frontend 경로는 테스트를 건너뛰고 바로 `summarizer`로 이동합니다.
> Planner 노드에서는 원래 human-in-the-loop interrupt가 발생하지만, 이 노트북에서는 이를 우회하여 직접 plan을 검사/수정할 수 있습니다.

## Step 1: Setup & Imports

프로젝트 루트를 `sys.path`에 추가하고, `.env`에서 환경변수를 로드하며,
`workspace/` 디렉토리를 생성합니다. 마지막으로 LLM에 테스트 메시지를 보내
연결 상태를 확인합니다.

- **`llm_config.get_llm()`**: `ChatOpenAI` 인스턴스 반환 (MODEL_NAME, temperature=0)
- **`llm_config.load_prompt()`**: `prompts/` 디렉토리에서 시스템 프롬프트 텍스트 로드

In [ ]:
import os, sys, copy, difflib, json, textwrap
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown, HTML

# --- 프로젝트 루트를 sys.path에 추가하여 로컬 모듈 import 가능하게 설정 ---
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- .env 파일에서 OPENAI_API_KEY, MODEL_NAME 등 환경변수 로드 ---
from dotenv import load_dotenv
load_dotenv()

# --- workspace/ 디렉토리 생성 (에이전트가 생성한 코드가 저장되는 곳) ---
Path("workspace").mkdir(exist_ok=True)

# --- LLM 연결 테스트: get_llm()이 ChatOpenAI 인스턴스를 반환하는지 확인 ---
from llm_config import get_llm, load_prompt
llm = get_llm()
test_resp = llm.invoke("Say 'OK' if you can hear me.")
print(f"LLM connected: {test_resp.content[:50]}")
print(f"Model: {os.getenv('MODEL_NAME', 'gpt-4o-mini')}")
print(f"Workspace: {Path('workspace').resolve()}")

## Step 2: Debug Helper Functions

디버깅에 사용할 유틸리티 함수들을 정의합니다. 각 함수의 역할:

| 함수 | 설명 |
|------|------|
| `show_state()` | state 필드를 마크다운 테이블로 출력 |
| `show_diff()` | 노드 실행 전후의 state 변경사항 표시 |
| `show_files()` | state에 저장된 코드 파일을 구문 강조와 함께 표시 |
| `show_logs()` | 실행 로그를 테이블로 출력 |
| `compare_code()` | 두 state 간의 코드 차이를 unified diff로 표시 |
| `apply_node_updates()` | LangGraph의 reducer 로직을 재현하여 state 업데이트 적용 |
| **`run_node()`** | **핵심 함수** — 노드 실행 + 자동 diff 표시 + 업데이트된 state 반환 |
| `show_prompt()` | LLM에 전송될 프롬프트 미리보기 |

In [ ]:
from state import AgentState, merge_dicts

def _truncate(val, maxlen=300):
    """긴 값을 maxlen 길이로 잘라서 표시용 문자열로 변환."""
    s = str(val)
    return s[:maxlen] + "..." if len(s) > maxlen else s

def show_state(state, fields=None, title="Current State"):
    """state의 지정된 필드들을 마크다운 테이블로 출력.
    fields를 지정하지 않으면 모든 키를 표시."""
    fields = fields or list(state.keys())
    lines = [f"### {title}\n", "| Field | Value |", "|-------|-------|"]
    for f in fields:
        val = state.get(f)
        lines.append(f"| `{f}` | {_truncate(val, 200)} |")
    display(Markdown("\n".join(lines)))

def show_diff(old_state, new_state, title="State Changes"):
    """노드 실행 전후 state를 비교하여 변경된 필드만 표시.
    변경이 없으면 '_No changes_'를 출력."""
    lines = [f"### {title}\n"]
    changed = False
    for key in set(list(old_state.keys()) + list(new_state.keys())):
        old_val = old_state.get(key)
        new_val = new_state.get(key)
        if old_val != new_val:
            changed = True
            lines.append(f"**`{key}`**: `{_truncate(old_val, 80)}` → `{_truncate(new_val, 80)}`\n")
    if not changed:
        lines.append("_No changes_")
    display(Markdown("\n".join(lines)))

def show_files(state):
    """state['files']의 모든 파일을 구문 강조된 코드 블록으로 표시."""
    files = state.get("files", {})
    if not files:
        print("No files in state.")
        return
    for fname, content in files.items():
        # 파일 확장자를 언어 이름으로 매핑 (구문 강조용)
        ext = Path(fname).suffix.lstrip(".")
        lang = {"py": "python", "js": "javascript", "html": "html"}.get(ext, ext)
        display(Markdown(f"#### 📄 {fname}\n```{lang}\n{content}\n```"))

def show_logs(state):
    """실행 로그를 번호가 매겨진 테이블로 표시.
    각 로그는 노드 이름과 해당 노드의 출력 요약을 포함."""
    logs = state.get("logs", [])
    if not logs:
        print("No logs yet.")
        return
    lines = ["### Execution Logs\n", "| # | Node | Details |", "|---|------|---------|"]
    for i, log in enumerate(logs):
        node = log.get("node", "?")
        details = ", ".join(f"{k}={_truncate(v, 60)}" for k, v in log.items() if k != "node")
        lines.append(f"| {i+1} | `{node}` | {details} |")
    display(Markdown("\n".join(lines)))

def compare_code(old_state, new_state):
    """두 state 간의 파일 변경사항을 unified diff로 표시.
    새로 추가된 파일, 수정된 파일 모두 diff로 출력."""
    old_files = old_state.get("files", {})
    new_files = new_state.get("files", {})
    all_files = set(list(old_files.keys()) + list(new_files.keys()))
    for fname in sorted(all_files):
        old = old_files.get(fname, "")
        new = new_files.get(fname, "")
        if old != new:
            diff = difflib.unified_diff(
                old.splitlines(keepends=True),
                new.splitlines(keepends=True),
                fromfile=f"{fname} (before)",
                tofile=f"{fname} (after)",
            )
            diff_text = "".join(diff)
            if diff_text:
                display(Markdown(f"#### Diff: {fname}\n```diff\n{diff_text}\n```"))

def apply_node_updates(state, updates):
    """노드 출력을 state에 적용. LangGraph의 reducer 로직을 재현:
    - files: merge_dicts로 병합 (기존 파일 유지 + 새 파일 추가/덮어쓰기)
    - logs: 리스트 append (operator.add reducer)
    - 나머지: 단순 덮어쓰기
    """
    new_state = copy.deepcopy(state)
    for key, val in updates.items():
        if key == "files":
            # files는 merge reducer: 기존 dict에 새 항목을 병합
            new_state["files"] = merge_dicts(new_state.get("files"), val)
        elif key == "logs":
            # logs는 add reducer: 리스트에 새 로그를 추가
            new_state["logs"] = new_state.get("logs", []) + val
        else:
            new_state[key] = val
    return new_state

def run_node(node_fn, state, node_name=None, **kwargs):
    """단일 노드를 실행하고 전후 diff를 표시한 뒤 업데이트된 state를 반환.
    이 함수가 노트북의 핵심 디버깅 워크플로우를 담당."""
    name = node_name or getattr(node_fn, "__name__", "unknown")
    print(f"{'='*60}")
    print(f"  Running: {name}")
    print(f"{'='*60}")
    
    # 실행 전 state를 deep copy하여 diff 비교에 사용
    old_state = copy.deepcopy(state)
    updates = node_fn(state, **kwargs)
    new_state = apply_node_updates(state, updates)
    
    # 변경된 필드만 하이라이트하여 표시
    show_diff(old_state, new_state, title=f"Changes from `{name}`")
    return new_state

def show_prompt(prompt_file, user_content):
    """LLM에 전송될 system prompt와 user message를 미리보기.
    실제 전송 전에 프롬프트 내용을 확인하여 디버깅에 활용."""
    system = load_prompt(prompt_file)
    display(Markdown(f"### LLM Prompt Preview\n\n**System prompt** (`{prompt_file}`):\n```\n{system[:500]}\n```\n\n**User message**:\n```\n{user_content[:500]}\n```"))

print("Debug helpers loaded ✓")

## Step 3: Initialize State

에이전트의 초기 state를 설정합니다. `GOAL`을 수정하여 원하는 작업을 지정하세요.

### AgentState 주요 필드
| 필드 | 타입 | 설명 |
|------|------|------|
| `goal` | str | 사용자가 입력한 코딩 목표 |
| `plan` | str | LLM이 생성한 구현 계획 (goal_analyzer → planner에서 설정) |
| `files` | dict | 생성된 코드 파일 (`{파일명: 내용}`, merge reducer 사용) |
| `test_code` | str | 생성된 pytest 코드 |
| `test_command` | str | 테스트 실행 명령어 (goal_analyzer가 추출) |
| `error_type` | str | 에러 분류: syntax / import / test_fail / runtime / logic / none |
| `diagnosis` | str | error_analyzer의 에러 진단 결과 |
| `rag_context` | str | RAG로 검색된 참고 코드 (선택적) |
| `iteration` | int | 현재 fix 반복 횟수 |
| `max_iterations` | int | 최대 반복 횟수 (기본값: 5) |
| `done` | bool | 파이프라인 완료 여부 |

In [ ]:
# ✏️ 여기를 수정하세요: 에이전트가 구현할 코딩 목표를 설정
GOAL = "Write a Python function that checks if a number is prime and returns True/False"

# AgentState와 동일한 구조로 초기 state 생성
state = {
    "goal": GOAL,               # 사용자의 자연어 코딩 목표
    "plan": "",                 # goal_analyzer → planner가 생성할 구현 계획
    "plan_approved": False,     # planner의 interrupt 승인 여부
    "files": {},                # 생성된 코드 파일 {파일명: 내용}
    "test_code": "",            # test_writer가 생성한 pytest 코드
    "test_command": "",         # goal_analyzer가 추출한 테스트 명령어
    "logs": [],                 # 각 노드의 실행 로그 (append reducer)
    "error_type": None,         # tester의 에러 분류 결과
    "error_message": None,      # tester의 에러 메시지 원문
    "diagnosis": None,          # error_analyzer의 에러 진단 결과
    "rag_context": None,        # rag_retriever가 검색한 참고 코드
    "iteration": 0,             # 현재 fix 반복 횟수
    "max_iterations": 5,        # 최대 반복 횟수 (무한 루프 방지)
    "done": False,              # 파이프라인 완료 플래그
}

show_state(state, title="Initial State")

## Step 4: Visualize the Graph

LangGraph StateGraph를 Mermaid 다이어그램으로 렌더링합니다.
10개의 노드와 조건부 라우팅 엣지를 확인할 수 있습니다.

확인할 포인트:
- `planner` → `backend_coder` / `frontend_coder` 분기
- `tester` → `error_analyzer` / `summarizer` 분기
- `fixer` → `tester` / `planner` 분기 (에러 유형에 따라)

> graphviz가 설치되어 있으면 PNG 이미지로, 없으면 Mermaid 텍스트로 출력됩니다.

In [ ]:
from graph import build_graph

# 컴파일된 LangGraph StateGraph 생성
graph = build_graph()

# 그래프 시각화: graphviz가 있으면 PNG, 없으면 Mermaid 텍스트로 출력
try:
    from IPython.display import Image
    png_data = graph.get_graph().draw_mermaid_png()
    display(Image(png_data))
except Exception:
    # graphviz 미설치 시 Mermaid 마크다운으로 대체 출력
    mermaid = graph.get_graph().draw_mermaid()
    display(Markdown(f"```mermaid\n{mermaid}\n```"))
    print("(Install graphviz + grandalf for visual rendering: pip install grandalf)")

---
## Step 5: Goal Analyzer

파이프라인의 **첫 번째 노드**입니다. 사용자의 자연어 목표를 분석하여:
1. **요구사항 목록**을 추출하고 `plan` 필드에 저장
2. **테스트 실행 명령어** (`TEST_COMMAND:` 마커)를 파싱하여 `test_command` 필드에 저장

프롬프트 파일: `prompts/analyzer.txt`

> 먼저 프롬프트 미리보기 셀을 실행하여 LLM에 전송될 내용을 확인한 후, 실제 노드를 실행하세요.

In [ ]:
# Goal Analyzer에 전송될 프롬프트 미리보기
# system prompt: prompts/analyzer.txt, user message: 사용자 목표
show_prompt("analyzer.txt", f"사용자의 목표: {state['goal']}")

In [ ]:
from nodes.goal_analyzer import goal_analyzer

# goal_analyzer 실행: goal → plan(요구사항) + test_command 추출
state = run_node(goal_analyzer, state)

# 핵심 출력 확인: 테스트 명령어와 추출된 요구사항
print(f"\nTest command: {state['test_command']}")
print(f"\nRequirements:\n{state['plan']}")

---
## Step 6: RAG Retriever (선택적)

기존 코드베이스에서 **FAISS 벡터 검색**을 통해 관련 코드를 검색합니다.
검색된 코드는 `rag_context`에 저장되어 이후 planner와 coder에 참고 자료로 전달됩니다.

- `CODEBASE_DIR` 환경변수가 설정되어 있어야 동작합니다
- 설정되지 않으면 gracefully skip하고 `rag_context = None`으로 유지됩니다
- 기존 프로젝트의 코드 패턴을 재활용하고 싶을 때 유용합니다

In [ ]:
from nodes.rag_retriever import rag_retriever

# CODEBASE_DIR 설정 여부 확인 (미설정 시 RAG를 건너뜀)
print(f"CODEBASE_DIR = {os.getenv('CODEBASE_DIR', '(not set)')}")

# rag_retriever 실행: FAISS 벡터 검색으로 관련 코드 검색
state = run_node(rag_retriever, state)

# 검색 결과 확인 (없으면 None으로 유지, 이후 단계에 영향 없음)
if state.get("rag_context"):
    print(f"\nRAG context ({len(state['rag_context'])} chars):")
    print(state["rag_context"][:500])
else:
    print("\nNo RAG context retrieved (CODEBASE_DIR not set or empty).")

---
## Step 7: Planner (interrupt 우회)

Planner 노드의 LLM 호출을 **직접 실행**하되, `interrupt()`를 **우회**합니다.
이것이 CLI 대비 이 노트북의 핵심 디버깅 장점입니다:

- CLI에서는 plan이 자동으로 승인/거부되지만, 여기서는 **plan을 직접 확인하고 수정**할 수 있습니다
- `plan_approved = True`를 수동으로 설정하여 interrupt 없이 다음 단계로 진행합니다
- 이전 에러 정보(`error_type`, `diagnosis`)가 있으면 프롬프트에 포함되어 re-planning에 활용됩니다

프롬프트 파일: `prompts/planner.txt`

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from tools.plan_ops import write_plan_md

# --- Planner와 동일한 프롬프트 구성 (interrupt 없이 직접 호출) ---
system_prompt = load_prompt("planner.txt")
user_content = f"목표: {state['goal']}\n요구사항:\n{state['plan']}"

# RAG 컨텍스트가 있으면 참고 코드로 추가 (최대 2000자)
if state.get("rag_context"):
    user_content += f"\n\n참고 코드:\n{state['rag_context'][:2000]}"

# 이전 에러가 있으면 re-planning을 위해 에러 정보 추가
if state.get("error_type") and state["error_type"] != "none":
    user_content += f"\n\n이전 에러 ({state['error_type']}): {state.get('diagnosis', '')}"
    user_content += f"\n반복 횟수: {state.get('iteration', 0)}"

# 프롬프트 미리보기
show_prompt("planner.txt", user_content)

# --- LLM 호출: plan 생성 ---
llm = get_llm()
result = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_content),
])
plan_text = result.content.strip()

# workspace/plan.md에 계획 저장
write_plan_md(plan_text)

# 생성된 plan 표시
display(Markdown(f"### Generated Plan\n\n{plan_text}"))

# --- State 업데이트 (interrupt 우회: plan_approved=True로 직접 설정) ---
old_state = copy.deepcopy(state)
state = apply_node_updates(state, {
    "plan": plan_text,
    "plan_approved": True,  # CLI에서는 interrupt로 사용자 승인을 받지만, 여기서는 자동 승인
    "logs": [{"node": "planner", "iteration": state.get("iteration", 0)}],
})
show_diff(old_state, state, title="Planner state changes")

### (선택) Plan 수동 편집

LLM이 생성한 plan이 마음에 들지 않을 때 직접 수정할 수 있습니다.
아래 셀의 주석을 해제하고 plan을 편집하세요.

수정이 유용한 경우:
- LLM이 불필요하게 복잡한 구현을 제안할 때
- 특정 라이브러리나 패턴을 강제하고 싶을 때
- 에러 수정 후 plan의 일부만 변경하고 싶을 때

In [ ]:
# ✏️ Plan을 수동으로 수정하려면 아래 주석을 해제하고 편집하세요:
# state["plan"] = """
# 1. Create is_prime(n) function
# 2. Handle edge cases: n <= 1 returns False
# 3. Check divisibility from 2 to sqrt(n)
# 4. Return True if no divisors found
# """
# write_plan_md(state["plan"])  # workspace/plan.md에도 반영
# print("Plan overridden!")

# 현재 적용된 plan 확인
print(f"Current plan:\n{state['plan']}")

---
## Step 8: Coder (Backend / Frontend 자동 라우팅)

목표(goal)에 포함된 키워드를 기반으로 **backend_coder** 또는 **frontend_coder**를 선택합니다.

**라우팅 로직** (`route_to_coder()`):
- Goal에 `html`, `css`, `javascript`, `react`, `웹페이지` 등의 키워드가 있으면 → `frontend_coder`
- 그 외 → `backend_coder` (기본값)

**출력 형식**:
- LLM 응답에서 `FILENAME: xxx.py` 마커와 코드 블록을 파싱
- `files` dict에 `{파일명: 코드}` 형태로 저장

> **주의**: Frontend 경로는 `done=True`를 설정하여 테스트를 건너뛰고 바로 Step 14(Summarizer)로 이동합니다.

In [ ]:
from nodes.coder import route_to_coder, FRONTEND_KEYWORDS
from nodes.backend_coder import backend_coder
from nodes.frontend_coder import frontend_coder

# --- 라우팅 결정: goal의 키워드로 backend/frontend 선택 ---
route = route_to_coder(state)
goal_lower = state["goal"].lower()
# 매칭된 프론트엔드 키워드 확인 (없으면 backend로 라우팅)
matched_kw = [kw for kw in FRONTEND_KEYWORDS if kw in goal_lower]
print(f"Routing decision: {route}")
print(f"Matched frontend keywords: {matched_kw if matched_kw else '(none → backend)'}")

# LLM에 전송될 프롬프트 미리보기
prompt_file = "backend_coder.txt" if route == "backend_coder" else "frontend_coder.txt"
user_content = f"계획:\n{state['plan']}"
if state.get("rag_context"):
    user_content += f"\n\n참고 코드:\n{state['rag_context'][:2000]}"
show_prompt(prompt_file, user_content)

In [ ]:
# 선택된 coder 노드 실행 (LLM 응답에서 FILENAME: + 코드 블록 파싱)
coder_fn = backend_coder if route == "backend_coder" else frontend_coder
state = run_node(coder_fn, state, node_name=route)

# 생성된 코드 파일 표시 (구문 강조)
show_files(state)

# Frontend 경로는 done=True를 설정하여 테스트를 건너뜀
if state.get("done"):
    print("\\n⚡ Frontend path: done=True. Skip to Step 14 (Summarizer).")

---
## Step 9: Test Writer

Coder가 생성한 소스 코드를 기반으로 **pytest 테스트 코드**를 자동 생성합니다.

- `state["files"]`의 **첫 번째 파일**을 대상으로 테스트를 작성합니다
- 생성된 테스트 코드는 `test_code` 필드와 `files` dict(예: `test_solution.py`)에 저장됩니다
- 프롬프트 파일: `prompts/test_writer.txt`

In [ ]:
from nodes.test_writer import test_writer

# state["files"]의 첫 번째 파일을 테스트 대상으로 선택
main_filename = list(state["files"].keys())[0]
main_code = state["files"][main_filename]

# Test Writer에 전송될 프롬프트 미리보기
tw_user = f"다음 코드를 테스트하는 pytest 코드를 작성하세요.\n\n코드 ({main_filename}):\n{main_code[:3000]}"
show_prompt("test_writer.txt", tw_user)

In [ ]:
# test_writer 실행: pytest 코드 생성 → test_code + files에 저장
state = run_node(test_writer, state)

# 소스 코드 + 테스트 코드 모두 표시
show_files(state)

---
## Step 10: Tester

테스트 명령어를 실행하고 결과를 **에러 유형별로 분류**합니다.

### 에러 분류 우선순위 (`classify_error()`)
```
returncode == 0  →  "none" (성공)
"SyntaxError"    →  "syntax"
"ModuleNotFound" →  "import"
"FAILED"         →  "test_fail"
"Traceback"      →  "runtime"
그 외             →  "logic"
```

### 이후 라우팅 (`route_after_tester()`)
- `done=True` 또는 `iteration >= max_iterations` → `summarizer` (종료)
- 그 외 → `error_analyzer` (에러 분석 시작)

> 첫 번째 셀은 raw 출력과 분류 과정을 보여주고, 두 번째 셀에서 실제 state를 업데이트합니다.

In [ ]:
from nodes.tester import tester, classify_error
from tools.exec_ops import run_command

print(f"Test command: {state['test_command']}")
print(f"Running in: {Path('workspace').resolve()}\n")

# --- 테스트 명령어 실행 (subprocess, 30초 타임아웃) ---
stdout, stderr, returncode = run_command(state["test_command"])
combined = stdout + "\n" + stderr

# Raw 출력 표시 (에러 분류 전 원본 확인용)
display(Markdown(f"### Raw Test Output\n\n**Return code:** `{returncode}`\n\n**stdout:**\n```\n{stdout[:1000]}\n```\n\n**stderr:**\n```\n{stderr[:1000]}\n```"))

# --- 에러 분류: 우선순위 기반으로 에러 유형 결정 ---
error_type = classify_error(combined, returncode)
print(f"\nError classification: {error_type}")

# 분류 과정의 각 체크 항목 표시 (디버깅용)
if returncode != 0:
    checks = {
        "SyntaxError in output": "SyntaxError" in combined,
        "ModuleNotFoundError/ImportError": "ModuleNotFoundError" in combined or "ImportError" in combined,
        "AssertionError/FAILED": "AssertionError" in combined or "AssertError" in combined or "FAILED" in combined,
        "Traceback": "Traceback" in combined,
    }
    print("Classification checks:")
    for check, matched in checks.items():
        print(f"  {'✓' if matched else '✗'} {check}")

In [ ]:
# tester 노드 실행: state에 error_type, error_message, done 업데이트
state = run_node(tester, state)

# 테스트 결과 요약
print(f"\nerror_type: {state['error_type']}")
print(f"done: {state['done']}")
if state.get("error_message"):
    print(f"error_message: {state['error_message'][:300]}")

# --- 라우팅 결정: 다음에 어떤 노드로 이동하는지 확인 ---
from graph import route_after_tester
next_node = route_after_tester(state)
print(f"\n→ Next node: {next_node}")
if next_node == "summarizer":
    # done=True(성공) 또는 max_iterations 도달 시 summarizer로 종료
    reason = "done=True (tests passed)" if state["done"] else f"iteration ({state['iteration']}) >= max_iterations ({state['max_iterations']})"
    print(f"  Reason: {reason}")

---
## Step 11: Error Analyzer (조건부)

테스트가 실패했을 때만 실행됩니다 (`error_type != "none"`).
LLM이 에러 메시지와 코드를 분석하여 **근본 원인을 한 문장으로 진단**합니다.

- 진단 결과는 `diagnosis` 필드에 저장되어 다음 단계(Fixer)에서 수정 가이드로 사용됩니다
- 테스트가 성공했다면 (`done=True`) 이 셀을 건너뛰고 Step 14로 이동하세요
- 프롬프트 파일: `prompts/error_analyzer.txt`

In [ ]:
from nodes.error_analyzer import error_analyzer

# 테스트 성공 시 이 셀을 건너뛰고 Step 14로 이동
if state.get("done"):
    print("Tests passed! No error to analyze. Skip to Step 14 (Summarizer).")
else:
    # --- Error Analyzer 프롬프트 구성: 에러 정보 + 전체 코드 ---
    files = state.get("files", {})
    code_text = ""
    for fname, content in files.items():
        # .md 파일(plan.md 등)은 제외하고 코드 파일만 포함
        if not fname.endswith(".md"):
            code_text += f"\n--- {fname} ---\n{content[:3000]}\n"
    ea_user = (
        f"에러 종류: {state.get('error_type', 'unknown')}\n"
        f"에러 메시지:\n{(state.get('error_message') or '')[:500]}\n\n"
        f"코드:{code_text}"
    )
    show_prompt("error_analyzer.txt", ea_user)
    
    # error_analyzer 실행: LLM이 에러 원인을 한 문장으로 진단
    state = run_node(error_analyzer, state)
    print(f"\nDiagnosis: {state['diagnosis']}")

---
## Step 12: Fixer (조건부)

Error Analyzer의 진단(`diagnosis`)을 기반으로 코드를 수정합니다.
수정 전후의 **unified diff**를 표시하여 변경사항을 확인할 수 있습니다.

### 수정 후 라우팅 (`route_after_fixer()`)
에러 유형에 따라 다음 경로가 결정됩니다:
- `syntax` / `import` → **tester로 직행** (빠른 재테스트, 계획 변경 불필요)
- `test_fail` / `runtime` / `logic` → **planner로 복귀** (재계획 필요)

> `iteration` 카운터가 자동으로 증가합니다. `max_iterations`에 도달하면 루프가 종료됩니다.

In [ ]:
from nodes.fixer import fixer
from graph import route_after_fixer

# 테스트 성공 시 이 셀을 건너뛰고 Step 14로 이동
if state.get("done"):
    print("Tests passed! No fix needed. Skip to Step 14 (Summarizer).")
else:
    # 수정 전 state 저장 (diff 비교용)
    old_state = copy.deepcopy(state)
    
    # fixer 실행: diagnosis를 기반으로 코드 수정 + iteration 증가
    state = run_node(fixer, state)
    
    # 수정 전후 코드 diff 표시
    compare_code(old_state, state)
    
    print(f"\nIteration: {state['iteration']} / {state['max_iterations']}")
    
    # --- 라우팅 결정: syntax/import → tester(빠른 재테스트), 그 외 → planner(재계획) ---
    next_after_fix = route_after_fixer(state)
    print(f"→ Next node after fixer: {next_after_fix}")
    if next_after_fix == "tester":
        print(f"  Reason: error_type='{state['error_type']}' is syntax/import -> fast re-test")
    else:
        print(f"  Reason: error_type='{state['error_type']}' needs re-planning")

---
## Step 13: Iteration Loop (test → analyze → fix 반복)

이 셀을 **재실행**하여 fix 루프를 반복할 수 있습니다.
각 반복에서 state, 코드 diff, 라우팅 결정을 자동으로 표시합니다.

### 루프 종료 조건
1. 테스트 통과 (`done=True`)
2. `MAX_LOOP` 횟수 도달 (이 셀 내부 제한)
3. `max_iterations` 도달 (전체 파이프라인 제한)
4. Fixer가 `planner`로 라우팅 → 루프를 빠져나와 **Step 7로 돌아가서 재계획** 필요

> `MAX_LOOP` 변수를 수정하여 이 셀 내의 반복 횟수를 조절할 수 있습니다.

In [ ]:
from nodes.tester import tester
from nodes.error_analyzer import error_analyzer
from nodes.fixer import fixer
from graph import route_after_tester, route_after_fixer

MAX_LOOP = 3  # ✏️ 이 셀 내의 최대 반복 횟수 조절

for loop_i in range(MAX_LOOP):
    display(Markdown(f"---\n### Loop iteration {loop_i + 1}"))
    
    # --- 1. Tester: 테스트 실행 + 에러 분류 ---
    old = copy.deepcopy(state)
    state = run_node(tester, state)
    
    next_node = route_after_tester(state)
    print(f"  error_type={state['error_type']}, done={state['done']}")
    print(f"  → route_after_tester: {next_node}")
    
    # 종료 조건: 테스트 통과 또는 max_iterations 도달
    if next_node == "summarizer":
        if state["done"]:
            print("\n✅ Tests passed! Exiting loop.")
        else:
            print(f"\n⛔ Max iterations reached ({state['iteration']}/{state['max_iterations']}). Exiting loop.")
        break
    
    # --- 2. Error Analyzer: 에러 원인 진단 ---
    state = run_node(error_analyzer, state)
    print(f"  Diagnosis: {state['diagnosis'][:200]}")
    
    # --- 3. Fixer: 진단 기반 코드 수정 ---
    old = copy.deepcopy(state)
    state = run_node(fixer, state)
    compare_code(old, state)
    
    # 라우팅 확인: tester로 재테스트 or planner로 재계획
    next_after_fix = route_after_fixer(state)
    print(f"  → route_after_fixer: {next_after_fix}")
    print(f"  Iteration: {state['iteration']}/{state['max_iterations']}")
    
    # Fixer가 planner로 라우팅하면 루프 탈출 (Step 7에서 재계획 필요)
    if next_after_fix == "planner":
        print("\n🔄 Fixer routes to planner. Go back to Step 7 to re-plan.")
        break

# 루프 종료 후 핵심 state 필드 요약
show_state(state, fields=["error_type", "error_message", "done", "iteration", "diagnosis"],
           title="State after loop")

---
## Step 14: Summarizer

실행 결과에서 **교훈(lesson)**을 한 문장으로 추출하고 `InMemoryStore`에 저장합니다.

- Store의 키는 타임스탬프 기반으로 생성됩니다
- 저장된 교훈은 이후 에이전트 실행에서 참고 자료로 활용될 수 있습니다
- 이 디버그 세션에서는 임시 `InMemoryStore`를 사용합니다 (세션 종료 시 소멸)
- 프롬프트 파일: `prompts/summarizer.txt`

In [ ]:
from nodes.summarizer import summarizer
from langgraph.store.memory import InMemoryStore

# 디버그 세션용 임시 Store 생성 (세션 종료 시 소멸)
debug_store = InMemoryStore()

# summarizer는 config(thread_id)와 store를 추가 인자로 필요로 함
dummy_config = {"configurable": {"thread_id": "debug"}}
state = run_node(
    summarizer, state, 
    node_name="summarizer",
    config=dummy_config, 
    store=debug_store,
)

# 마지막 로그에서 추출된 교훈(lesson) 확인
last_log = state["logs"][-1] if state["logs"] else {}
print(f"\nLesson: {last_log.get('lesson', 'N/A')}")

---
## Step 15: Final Summary

파이프라인 실행의 전체 결과를 한눈에 확인합니다:

1. **실행 로그**: 모든 노드의 실행 기록 (테이블)
2. **최종 State**: goal, done, iteration, error_type 등 핵심 필드
3. **생성된 코드**: state에 저장된 모든 파일 (구문 강조)
4. **Workspace 파일**: 디스크에 실제로 저장된 파일 목록

In [ ]:
# --- 1. 전체 실행 로그 (모든 노드의 실행 기록) ---
show_logs(state)

# --- 2. 최종 state 요약 (핵심 필드만) ---
show_state(state, fields=["goal", "done", "iteration", "error_type", "test_command"],
           title="Final State Summary")

# --- 3. 생성된 코드 파일 (구문 강조) ---
display(Markdown("### Generated Files"))
show_files(state)

# --- 4. 디스크에 저장된 workspace 파일 목록 ---
from tools.file_ops import list_files
workspace_files = list_files()
print(f"\nWorkspace files on disk: {workspace_files}")

---
## Alternative: Full Pipeline Run (그래프 전체 실행)

위의 step-by-step 방식과 달리, **컴파일된 LangGraph를 `stream()`으로 한번에 실행**합니다.
실제 CLI(`main.py`)와 동일한 방식으로 동작하며, 모든 이벤트를 캡처합니다.

**Step-by-step과의 차이점**:
- 실제 그래프의 라우팅 로직과 interrupt 처리가 동작합니다
- Planner의 interrupt는 **자동 승인**(auto-approve)으로 처리됩니다
- `Command(resume={"action": "approve"})`를 `"modify"` 또는 `"reject"`로 변경하여 다른 경로를 테스트할 수 있습니다

In [ ]:
from graph import build_graph
from checkpointer import get_thread_config
from langgraph.types import Command

# --- 그래프 및 체크포인터 설정 ---
full_graph = build_graph()
full_config = get_thread_config()  # thread_id 기반 체크포인트 config

# ✏️ 목표 설정 (위의 GOAL과 동일하거나 별도로 지정 가능)
full_goal = GOAL

# 초기 state (Step 3과 동일한 구조)
full_initial = {
    "goal": full_goal,
    "plan": "", "plan_approved": False, "files": {}, "test_code": "",
    "test_command": "", "logs": [], "error_type": None, "error_message": None,
    "diagnosis": None, "rag_context": None, "iteration": 0, "max_iterations": 5,
    "done": False,
}

# --- 그래프 스트리밍 실행 (이벤트 캡처) ---
all_events = []
input_data = full_initial

while True:
    has_interrupt = False
    
    # stream()으로 그래프 실행, 각 노드의 업데이트를 이벤트로 수신
    for event in full_graph.stream(input_data, config=full_config, stream_mode="updates"):
        all_events.append(event)
        for node_name, updates in event.items():
            print(f"\n--- [{node_name}] ---")
            if isinstance(updates, dict) and "logs" in updates:
                for log in updates["logs"]:
                    for k, v in log.items():
                        if k != "node":
                            print(f"  {k}: {str(v)[:200]}")
            if isinstance(updates, dict) and updates.get("done"):
                print("  ✅ SUCCESS!")
    
    # --- Interrupt 처리: planner 노드의 human-in-the-loop 승인 ---
    graph_state = full_graph.get_state(full_config)
    if graph_state.tasks:
        for task in graph_state.tasks:
            if hasattr(task, "interrupts") and task.interrupts:
                # Interrupt에서 plan 데이터 추출
                plan_data = task.interrupts[0].value
                display(Markdown(f"### Plan for Approval\n\n{plan_data.get('plan', '')}"))
                # ✏️ 자동 승인: "modify" 또는 "reject"로 변경하여 다른 경로 테스트 가능
                input_data = Command(resume={"action": "approve"})
                has_interrupt = True
                print("→ Auto-approved plan")
                break
    
    # Interrupt가 없으면 파이프라인 완료
    if not has_interrupt:
        break

display(Markdown(f"### Pipeline complete — {len(all_events)} events captured"))

---
## Inspect Captured Events (이벤트 브라우저)

위의 Full Pipeline Run에서 캡처한 이벤트를 하나씩 탐색합니다.

- 각 이벤트는 `{노드명: 업데이트된_state_필드}` 형태의 dict입니다
- `EVENT_INDEX`를 변경하여 원하는 이벤트를 확인하세요
- 이벤트 목록과 노드 순서도 함께 출력됩니다

In [ ]:
# ✏️ 탐색할 이벤트 인덱스 (0-based, 위의 Full Pipeline 실행 결과에서 선택)
EVENT_INDEX = 0  

if all_events:
    if EVENT_INDEX < len(all_events):
        event = all_events[EVENT_INDEX]
        # 각 이벤트는 {노드명: 업데이트된_필드들} 형태의 dict
        for node_name, data in event.items():
            display(Markdown(f"### Event {EVENT_INDEX}: `{node_name}`"))
            if isinstance(data, dict):
                for k, v in data.items():
                    val_str = str(v)
                    if len(val_str) > 500:
                        val_str = val_str[:500] + "..."
                    print(f"  {k}: {val_str}")
    else:
        print(f"Index {EVENT_INDEX} out of range. Total events: {len(all_events)}")
    
    # 전체 이벤트 개수와 노드 실행 순서 표시
    print(f"\nTotal events: {len(all_events)}")
    print("Event nodes:", [list(e.keys())[0] for e in all_events])
else:
    print("No events captured. Run the full pipeline cell first.")

---\n## Debugging Tips\n\n**Step-by-step mode (Steps 5-14):**\n- Run cells one by one, inspect state after each\n- Edit `state` dict directly between cells to test specific scenarios\n- Override the plan in Step 7 to control what gets coded\n- Re-run Step 13 loop cell multiple times for fix iterations\n\n**Full pipeline mode (Alternative section):**\n- Change `Command(resume={"action": "approve"})` to `"modify"` or `"reject"` to test interrupt paths\n- Browse events with the event inspector cell\n\n**Common debugging scenarios:**\n1. **LLM generates bad code** → Check prompt in the Preview cells, try editing the plan\n2. **Wrong error classification** → Step 10 shows raw output + classification checks\n3. **Fixer makes things worse** → Compare diffs in Step 12, check diagnosis accuracy\n4. **Infinite loop** → Check routing decisions at each step (shown automatically)\n5. **RAG not working** → Step 6 shows CODEBASE_DIR status and retrieved chunks